# Predictive Maintenance of Rotating Machinery - Analysis Notebook
**All data is simulated.** Run from the `notebooks/` folder after `pip install -r requirements.txt`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from data_generator import generate_dataset
from features import add_features, FEATURE_COLUMNS
from fault_detection import evaluate_dataframe
from health_score import health_scores
from ml_model import train_and_evaluate

## 1. Generate and inspect the simulated data

In [ ]:
df = generate_dataset()
print(df.shape)
df.head()

In [ ]:
df.groupby('fault_type')[['rpm','temperature','vibration_rms','current']].mean().round(2)

## 2. Sensor trends in one bearing-degradation run

In [ ]:
e = df[df.fault_type=='bearing_degradation']
e = e[e.episode_id==e.episode_id.iloc[0]].reset_index(drop=True)
e[['temperature','vibration_rms','current']].plot(subplots=True, figsize=(9,6));

## 3. Feature extraction (moving averages, rates of change, rolling RMS)

In [ ]:
f = add_features(df)
f[FEATURE_COLUMNS].describe().T.round(3)

## 4. Rule-based detection and health score

In [ ]:
r = evaluate_dataframe(df)
r['health'], _ = health_scores(r)
print(pd.crosstab(r.machine_condition, r.rule_status))
r.groupby('machine_condition').health.describe().round(1)

## 5. Random Forest (episode-wise split, no leakage)

In [ ]:
m = train_and_evaluate(df, save=False)
{k: round(m[k],3) for k in ['accuracy','precision_macro','recall_macro','cv_accuracy_mean','rule_based_accuracy']}

In [ ]:
print(pd.DataFrame(m['confusion_matrix'], index=m['labels'], columns=m['labels']))
pd.Series(m['feature_importance']).sort_values().plot.barh(figsize=(6,4));

**Caveat:** both the rules and the model are evaluated against labels created by the simulator, so these numbers show the pipeline works - they do not prove real-world performance.